# 🌿 KisanMitra AI — Notebook 02: CNN Disease Detection Training & Grad-CAM

**Course Project:** B.Tech AI/ML  
**Objective:** Train MobileNetV3-Small baseline, train EfficientNet-B0, compare on test metrics, analyze confusion matrix, and generate Grad-CAM visual heatmaps.  
**Hardware:** Designed for Google Colab (Free T4 GPU), Kaggle Notebooks, or Local (Apple Silicon / CPU).

## 1. Hardware & Environment Check

In [ ]:
import os
import sys
from pathlib import Path
import torch

# Check device
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Active Device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Clone or navigate to Aikissan repo
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('Aikissan'):
    !git clone https://github.com/namangoyal176-10/Aikissan.git
    %cd Aikissan
    sys.path.insert(0, os.getcwd())
else:
    project_root = Path.cwd().resolve()
    if project_root.name == 'notebooks':
        os.chdir(project_root.parent)
    sys.path.insert(0, str(Path.cwd()))

print("Working directory:", os.getcwd())

## 2. Safe Kaggle Credential Handling (NEVER Hard-Code Keys!)

> **Security Rule:** Never type your Kaggle username or key directly into code cells. Notebooks can accidentally be made public or committed to GitHub.

Choose **Option A** (Colab Secrets) or **Option B** (File Upload):

In [ ]:
# OPTION A: Google Colab Secrets (Recommended for Colab)
# 1. Click the 'Key' icon (Secrets) in Colab's left sidebar
# 2. Add 'KAGGLE_USERNAME' and 'KAGGLE_KEY'
# 3. Toggle 'Notebook access' to ON

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✓ Kaggle credentials safely loaded from Colab Secrets!")
except Exception:
    print("ℹ️ Colab secrets not detected. If running locally or on Kaggle, use existing environment.")

# OPTION B: Interactive Upload (If secrets not available)
# from google.colab import files
# uploaded = files.upload()  # upload your kaggle.json here
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 3. Verify PlantVillage Dataset & Split Manifests

Checks the 13 Potato and Tomato folders and generates deterministic 70/15/15 stratified train/val/test splits.

In [ ]:
# Run dataset verification and splitting
!python3 src/disease/prepare_dataset.py

## 4. Sanity Check Run (50 Images/Class, 2 Epochs)

Before committing GPU hours to a full run, we test the entire pipeline on a tiny subset.
This catches shape mismatches, memory leaks, and logging bugs in under 60 seconds.

In [ ]:
!python3 src/disease/train.py \
    --model-name mobilenet_v3_small \
    --sanity-check \
    --batch-size 32 \
    --lr 0.001

## 5. Full Training Run: Model 1 — MobileNetV3-Small (Baseline)

- **Architecture:** MobileNetV3-Small (~2.5M parameters)
- **Augmentations:** RandomResizedCrop, Rotation, ColorJitter, GaussianBlur
- **Loss:** Class-Weighted Cross-Entropy Loss
- **Optimizer:** AdamW with CosineAnnealingLR

In [ ]:
!python3 src/disease/train.py \
    --model-name mobilenet_v3_small \
    --epochs 15 \
    --batch-size 32 \
    --lr 0.001 \
    --weight-decay 0.0001

## 6. Full Training Run: Model 2 — EfficientNet-B0 (Advanced Transfer Learning)

- **Architecture:** EfficientNet-B0 (~5.3M parameters)
- Uses Compound Scaling for superior feature representation.

In [ ]:
!python3 src/disease/train.py \
    --model-name efficientnet_b0 \
    --epochs 15 \
    --batch-size 32 \
    --lr 0.0003 \
    --weight-decay 0.0001

## 7. Comparative Evaluation on Test Split (70/15/15 Holdout)

Evaluates both models on the unseen test split and compares:
1. Test Accuracy & Macro F1
2. Model Size (MB) & Parameter Count
3. CPU Inference Latency (ms/image)

In [ ]:
# Evaluate MobileNetV3-Small
!python3 src/disease/evaluate.py --checkpoint models/mobilenet_v3_small_best.pt

# Evaluate EfficientNet-B0
!python3 src/disease/evaluate.py --checkpoint models/efficientnet_b0_best.pt

## 8. Grad-CAM Visual Explainability

Visualize the Grad-CAM heatmap overlay for a sample test image to confirm the model attends to disease lesions rather than image backgrounds.

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from src.disease.gradcam import explain_leaf_image
from src.disease.model import get_disease_model
from src.utils.config import ALL_DISEASE_CLASSES

# Pick a test image from test manifest
test_df = pd.read_csv("data/processed/disease_test.csv")
sample_row = test_df.iloc[0]
sample_img_path = sample_row["image_path"]
true_label = sample_row["class_name"]

# Load MobileNet checkpoint
model = get_disease_model("mobilenet_v3_small", num_classes=len(ALL_DISEASE_CLASSES), pretrained=False)
chkpt = torch.load("models/mobilenet_v3_small_best.pt", map_location="cpu")
model.load_state_dict(chkpt["model_state_dict"])

pred_label, conf, blended_img, heatmap = explain_leaf_image(
    model=model,
    model_name="mobilenet_v3_small",
    image_path=sample_img_path,
    device="cpu",
    output_path="reports/sample_gradcam.jpg",
)

# Plot original vs Grad-CAM
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(sample_img_path))
axes[0].set_title(f"True: {true_label}")
axes[0].axis("off")

axes[1].imshow(blended_img)
axes[1].set_title(f"Grad-CAM: {pred_label} ({conf*100:.1f}%)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 9. Safe Storage of Model Weights (Keep .pt Out of Git!)

> **Git Rule Reminder:** Model checkpoints are large binary files (>10MB). They should **never** be committed to Git.

### Recommended Options:
1. **Download to Local Project:**
   ```python
   from google.colab import files
   files.download('models/mobilenet_v3_small_best.pt')
   files.download('models/efficientnet_b0_best.pt')
   ```
   Place the downloaded files into your local `models/` directory.

2. **Hugging Face Hub (Industry Standard for Portfolios):**
   ```python
   # !pip install huggingface_hub
   # from huggingface_hub import HfApi
   # api = HfApi()
   # api.upload_file(
   #     path_or_fileobj='models/mobilenet_v3_small_best.pt',
   #     path_in_repo='mobilenet_v3_small_best.pt',
   #     repo_id='your_hf_username/kisanmitra-models',
   #     repo_type='model',
   # )
   ```

3. **Google Drive Backup:**
   ```python
   # from google.colab import drive
   # drive.mount('/content/drive')
   # !cp models/*_best.pt /content/drive/MyDrive/KisanMitra_Models/
   ```